# अध्याय ७: च्याट एप्लिकेसनहरू निर्माण गर्ने
## OpenAI API द्रुत आरम्भ

यो नोटबुक [Azure OpenAI नमूना भण्डार](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) बाट अनुकूलित गरिएको हो जसमा नोटबुकहरू छन् जुन [Azure OpenAI](notebook-azure-openai.ipynb) सेवाहरूमा पहुँच गर्छन्।

Python OpenAI API Azure OpenAI मोडेलहरूसँग पनि काम गर्छ, थोरै संशोधनहरूका साथ। यहाँ फरकहरूबारे थप जान्नुस्: [Python सँग OpenAI र Azure OpenAI अन्त बिन्दुहरू बीचमा कसरी स्विच गर्ने](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# अवलोकन  
"ठूला भाषा मोडेलहरू फङ्सनहरू हुन् जसले पाठलाई पाठमा नक्सांकन गर्छन्। एउटा इनपुट स्ट्रिङ पाठ दिइएपछि, एउटा ठूलो भाषा मोडेलले आगामी पाठ के आउँछ अनुमान लगाउने प्रयास गर्छ"(1)। यो "शुरु गर्ने" नोटबुकले प्रयोगकर्ताहरूलाई उच्च-स्तरीय LLM अवधारणाहरू, AML सँग सुरु गर्न आवश्यक मुख्य प्याकेजहरू, प्रॉम्प्ट डिजाइनको दुईआम परिचय, र विभिन्न प्रयोग केसहरूको केही छोटो उदाहरणहरू परिचय गर्नेछ। 


## सूची तालिका  

[संक्षिप्त अवलोकन](#overview)  
[OpenAI सेवा कसरी प्रयोग गर्ने](#how-to-use-openai-service)  
[1. आफ्नो OpenAI सेवा सिर्जना गर्ने](#1.-creating-your-openai-service)  
[2. स्थापना](#2.-installation)    
[3. प्रमाणपत्रहरू](#3.-credentials)  

[प्रयोग केसहरू](#use-cases)    
[1. पाठ सारांश गर्नुहोस्](#1.-summarize-text)  
[2. पाठ वर्गीकरण गर्नुहोस्](#2.-classify-text)  
[3. नयाँ उत्पादन नामहरू सिर्जना गर्नुहोस्](#3.-generate-new-product-names)  
[4. वर्गीकरणलाई सूक्ष्म समायोजन गर्नुहोस्](#4.fine-tune-a-classifier)  

[सन्दर्भहरू](#references)


### आफ्नो पहिलो प्रॉम्प्ट बनाउनुहोस्  
यो छोटो अभ्यासले एक साधारण कार्य "सारांश" को लागि OpenAI मोडेलमा प्रॉम्प्ट कसरी प्रस्तुत गर्ने आधारभूत परिचय प्रदान गर्नेछ।  


**चरणहरू**:  
1. आफ्नो पायथन वातावरणमा OpenAI पुस्तकालय स्थापना गर्नुहोस्  
2. मानक सहायक पुस्तकालयहरू लोड गर्नुहोस् र तपाईंले सिर्जना गरेको OpenAI सेवाको लागि सामान्य OpenAI सुरक्षा प्रमाणपत्रहरू सेट गर्नुहोस्  
3. आफ्नो कार्यका लागि मोडेल चयन गर्नुहोस्  
4. मोडेलका लागि एक सरल प्रॉम्प्ट सिर्जना गर्नुहोस्  
5. मोडेल API लाई तपाईंको अनुरोध प्रस्तुत गर्नुहोस्!  


### १. OpenAI स्थापना गर्नुहोस्


In [ ]:
%pip install openai python-dotenv

### 2. सहायक पुस्तकालयहरू आयात गर्नुहोस् र प्रमाणपत्रहरू सुरु गर्नुहोस्


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. सही मोडल फेला पार्ने  
GPT-4o र GPT-4o मिनी जस्ता मोडेलहरूले प्राकृतिक भाषा बुझ्न र उत्पादन गर्न सक्छन्, र OpenAI प्लेटफर्ममा विभिन्न कार्यहरूको लागि उपयुक्त शक्ति र गतिमा उपलब्ध छन्।  


In [ ]:
# Select a general purpose chat model
model = "gpt-4o-mini"


## ४. प्रॉम्प्ट डिजाइन  

"ठूलो भाषा मोडेलहरूको जादू यो हो कि विशाल मात्रामा पाठमा यो पूर्वानुमान त्रुटि न्यूनतम बनाउन तालिम दिइएपछि, मोडेलहरूले यी पूर्वानुमानहरूका लागि उपयोगी अवधारणाहरू सिक्छन्। उदाहरणका लागि, तिनीहरूले यस्ता अवधारणाहरू सिक्छन्"(1):

* कसरि शब्दहरू स्पेल गर्ने
* कसरि व्याकरण काम गर्छ
* कसरि पैराफ्रेज गर्ने
* कसरि प्रश्नहरूको उत्तर दिने
* कसरि संवाद राख्ने
* धेरै भाषाहरूमा लेख्ने
* कसरि कोड लेख्ने
* आदि।

#### ठूलो भाषा मोडेल नियन्त्रण गर्ने तरिका  
"ठूलो भाषा मोडेलमा सबै इनपुटहरूसँग तुलनात्मक रूपमा सबैभन्दा प्रभावकारी भनेको पाठ प्रॉम्प्ट हो(1)।

ठूलो भाषा मोडेलहरूलाई केही तरिकाले आउटपुट उत्पादन गर्न प्रॉम्प्ट गर्न सकिन्छ:

निर्देशन: मोडेललाई तपाईँ के चाहनुहुन्छ भन्नुहोस्
पूर्णता: तपाईँले चाहेको कुरा सुरु गराउन मोडेललाई प्रेरित गर्नुहोस्
प्रदर्शन: मोडेललाई तपाईँ के चाहनुहुन्छ देखाउनुहोस्, या त:
प्रॉम्प्टमा केहि उदाहरणहरू
राम्रो तालिम डेटासेटमा सयौं वा हजारौं उदाहरणहरू"



#### प्रॉम्प्ट सिर्जना गर्ने तीन आधारभूत निर्देशनहरू छन्:

**देखाउनुहोस् र भन्नुहोस्।** तपाईँ के चाहनुहुन्छ भने प्रष्ट पार्नुहोस् निर्देशनहरू, उदाहरणहरू, वा दुवैको संयोजन मार्फत। यदि तपाईँ मोडेललाई वस्तुहरूको सूची वर्णानुक्रममा क्रमबद्ध गर्न वा एउटा अनुच्छेदलाई भावना अनुसार वर्गीकरण गर्न चाहनुहुन्छ भने, यो नै चाहिएको कुरा देखाउनुहोस्।

**गुणस्तरीय डाटा प्रदान गर्नुहोस्।** यदि तपाईँ क्लासिफायर बनाउन खोज्दै हुनुहुन्छ वा मोडेललाई कुनै ढाँचामा चले जान भन्न चाहनुहुन्छ भने, पर्याप्त उदाहरणहरू हुनु आवश्यक छ। तपाईँका उदाहरणहरूको प्रूफरीडिङ्ग गर्न निश्चित हुनुहोस् — मोडेल प्रायः साधारण स्पेलिङ गल्तीहरू देख्न पर्याप्त स्मार्ट हुन्छ र जवाफ दिन्छ, तर यसलाई उद्देश्यपूर्ण ठानेर जवाफमा असर पर्न सक्छ।

**सेटिङहरू जाँच गर्नुहोस्।** तापक्रम र top_p सेटिङहरूले मोडेल कति निर्धारणात्मक जवाफ उत्पन्न गर्दछ भन्ने नियन्त्रण गर्छ। यदि तपाईँले एक मात्र सही उत्तर चाहनुहुन्छ भने, यी सेटिङहरू कम राख्न चाहनुहुन्छ। यदि तपाईंले विभिन्न प्रकारका जवाफहरू खोज्दै हुनुहुन्छ भने, यी बढी राख्न चाहनुहुन्छ। यी सेटिङहरूसँग सबैभन्दा ठूलो गल्ती भनेको यिनीहरूलाई "चतुराई" वा "रचनात्मकता" नियन्त्रणहरू भन्ने अनुमान गर्नु हो।


स्रोत: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. पेश गर्नुहोस्!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### उही कल दोहोर्याउनुहोस्, नतिजाहरू कसरी तुलना गरिन्छ?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## पाठ सारांश  
#### चुनौती  
पाठको अन्त्यमा 'tl;dr:' थपेर लेखलाई संक्षेप गर्नुहोस्। मोडलले अतिरिक्त निर्देशन बिना धेरै कार्यहरू कसरी गर्ने बुझ्छ भन्ने कुरामा ध्यान दिनुहोस्। मोडलको व्यवहार परिवर्तन गर्न र प्राप्त सारांशलाई अनुकूलित गर्न tl;dr भन्दा थप वर्णनात्मक प्रम्प्टहरूसँग प्रयोग गर्न सक्नुहुन्छ(3)।  

हालैका कामहरूले धेरै NLP कार्यहरू र मानकहरूको क्षेत्रमा ठूलो टेक्स्ट कर्पसमा प्रि-ट्रेनिंग गरेर त्यसपछि विशिष्ट कार्यमा फाइन-ट्यूनिंग गरेर उल्लेखनीय प्रगति देखाएका छन्। सामान्यतया आर्किटेक्चरमा कार्य-अपेक्षित भएता पनि, यस पद्धतिले हजारौं वा दसौं हजार उदाहरणहरू भएको कार्य-विशिष्ट फाइन-ट्यूनिंग डेटासेटहरू आवश्यक पर्छ। यसको विपरीत, मानिसहरू सामान्यतया केही उदाहरणहरू वा सरल निर्देशनहरूबाट नयाँ भाषा कार्य गर्न सक्छन् - जुन वर्तमान NLP प्रणालीहरूले अझै पनि धेरै हदसम्म गर्न सक्दैन। यहाँ हामी देखाउँछौं कि भाषा मोडलहरूलाई स्केल गर्नुले कार्य-अपेक्षित, थोरै-शट प्रदर्शनलाई ठूलो सुधार गर्दछ, कहिलेकाहीं अघिल्लो अत्याधुनिक फाइन-ट्यूनिंग दृष्टिकोणहरूसँग प्रतिस्पर्धी पनि हुन सक्छ।  



संक्षेप  


# विभिन्न प्रयोगका लागि अभ्यासहरू  
1. पाठको सारांश बनाउनुहोस्  
2. पाठ वर्गीकृत गर्नुहोस्  
3. नयाँ उत्पादन नामहरू सिर्जना गर्नुहोस्


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## पाठ वर्गीकरण गर्नुहोस्  
#### चुनौती  
वस्तुहरूलाई अनुमान लगाउने समयमा दिइएका वर्गहरूमा वर्गीकृत गर्नुहोस्। तलको उदाहरणमा, हामीले वर्गहरू र वर्गीकृत गर्नुपर्ने पाठ दुवै प्रम्प्टमा प्रदान गरेका छौं(*playground_reference)। 

ग्राहक सोधपुछ: नमस्ते, मेरो ल्यापटपको कीबोर्डको एउटा कुञ्जी भर्खरै टुट्यो र मलाई यसको प्रतिस्थापन चाहिन्छ:

वर्गीकृत वर्ग:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## नयाँ उत्पादन नामहरू सिर्जना गर्नुहोस्
#### चुनौती
उदाहरण शब्दहरूबाट उत्पादन नामहरू सिर्जना गर्नुहोस्। यहाँ हामीले नामहरू सिर्जना गर्न लागेको उत्पादनको बारेमा जानकारी पनि प्रस्तावमा समावेश गरेका छौं। हामीले समान उदाहरण पनि दिएका छौं जसले हामीले पाउनु पर्ने ढाँचालाई देखाउँछ। हामीले तातो मान (temperature) पनि उच्च राखेका छौं जसले अनियमितता र अझ नवप्रवर्तनशील प्रतिक्रियाहरू बढाउन मद्दत गर्छ।

उत्पादन विवरण: घरमा प्रयोग हुने दूधशेक मेकर
बीउ शब्दहरू: छिटो, स्वस्थ, सानो।
उत्पादन नामहरू: HomeShaker, Fit Shaker, QuickShake, Shake Maker

उत्पादन विवरण: कुनै पनि खुट्टाको आकारमा मिल्ने जुत्ता जोडी।
बीउ शब्दहरू: अनुकूल, फिट, ओम्नी-फिट।


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# सन्दर्भहरू  
- [Openai कुकबुक](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry पोर्टल](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [पाठ वर्गीकरण गर्न GPT मोडेलहरूलाई राम्रोसँग ट्युनिङ गर्ने उत्तम अभ्यासहरू](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# थप मद्दतको लागि  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# योगदानकर्ताहरू
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
